# [IAPR][iapr]: Final project - Chocolate Recognition


**Moodle group ID:** *64*  
**Kaggle challenge:** *Classic*   
**Kaggle team name (exact):** "*Team Nah!*"  

**Author 1 (sciper):** Andrew Brown (370751)  
**Author 2 (sciper):** Nour Lachat (302397)   
**Author 3 (sciper):** Henry Farrell (402247) 

**Due date:** 21.05.2025 (11:59 pm)


## Key Submission Guidelines:
- **Before submitting your notebook, <span style="color:red;">rerun</span> it from scratch!** Go to: `Kernel` > `Restart & Run All`
- **Only groups of three will be accepted**, except in exceptional circumstances.


[iapr]: https://github.com/LTS5/iapr2025

---

> Your comments  
> ...

In [1]:
import src.lab1 as lab1 
import src.lab_01_utils as lab1utils
import src.lab2 as lab2
import src.lab_02_utils as lab2utils
import src.lab3 as lab3
import src.lab_03_utils as lab3utils
import src.preprocessing as pre
import matplotlib.pyplot as plt
import numpy as np
from sklearn.cluster import MiniBatchKMeans
from skimage.morphology import remove_small_objects, remove_small_holes, closing, disk, opening
from skimage.transform import rotate, resize
from sklearn.metrics.pairwise import euclidean_distances
from skimage.measure import regionprops
import cv2
from typing import Literal, Callable
import math


In [2]:
# Load the Datasets
references_path = "../data/dataset_project_iapr2025/references"
reference_images_raw, reference_labels, reference_dict = pre.load_reference_images(references_path)


train_path = "../data/dataset_project_iapr2025/train"
train_images_raw, train_labels, train_dict = pre.load_train_images(train_path)

In [ ]:
# Print Shape of Images
print(f"Reference images shape: {reference_images_raw[0].shape}")
print(f"Train images shape: {train_images_raw[0].shape}")

In [4]:
reference_images = reference_images_raw.copy()
train_images = train_images_raw.copy()

# Downsample the images and update the reference_images list and dictionary
for i, imgs in enumerate(reference_images):
    # Downsample the image using grid method
    downsampled_grid = pre.downsample_image(
        imgs,
        method="grid",
        target_height=400,
        target_width=600  # Only one dimension needs specification
    )

    # Update the reference_images list with the downsampled images
    reference_images[i] = downsampled_grid
    reference_dict[reference_labels[i]] = downsampled_grid


# Do the same with the train images
for i, imgs in enumerate(train_images):
    # Downsample the image using grid method
    downsampled_grid = pre.downsample_image(
        imgs,
        method="grid",
        target_height=400,
        target_width=600  # Only one dimension needs specification
    )

    # Update the reference_images list with the downsampled images
    train_images[i] = downsampled_grid
    train_dict[train_labels[i]] = downsampled_grid



In [ ]:
# Print New Image Shape vs Old Image Shape
print(f"Old Image Shape: {train_images_raw[0].shape}")
print(f"New Image Shape: {train_images[0].shape}")


In [6]:

# def joint_kmeans_downsample(
#     images: list[np.ndarray],
#     n_colors: int = 256
# ) -> tuple[list[np.ndarray], np.ndarray]:
#     """
#     Applies k-means color quantization jointly across all images and returns both the quantized images
#     and the learned color profile (cluster centers).
    
#     Args:
#         images: List of RGB images (H, W, 3)
#         n_colors: Number of color clusters to learn
    
#     Returns:
#         quantized_images: List of images with reduced color palette
#         color_profile: Array of shape (n_colors, 3), the shared color centers
#     """
#     # Flatten and concatenate all pixels from all images
#     pixel_arrays = [img.reshape(-1, 3) for img in images]
#     all_pixels = np.concatenate(pixel_arrays, axis=0)

#     # Fit KMeans to all image pixels
#     kmeans = MiniBatchKMeans(n_clusters=n_colors, random_state=0, batch_size=1024)
#     kmeans.fit(all_pixels)

#     # Quantize each image using the learned centers
#     quantized_images = []
#     for img in images:
#         flat_img = img.reshape(-1, 3)
#         labels = kmeans.predict(flat_img)
#         quantized = kmeans.cluster_centers_[labels].reshape(img.shape).astype(np.uint8)
#         quantized_images.append(quantized)

#     return quantized_images, kmeans.cluster_centers_.astype(np.uint8)

# def plot_color_profile(color_profile: np.ndarray, swatch_size: int = 2, cols: int = 32):
#     """
#     Plots a grid of color swatches from the color profile.

#     Args:
#         color_profile (np.ndarray): Array of shape (n_colors, 3)
#         swatch_size (int): Size of each square swatch in pixels
#         cols (int): Number of columns in the grid
#     """
#     n_colors = color_profile.shape[0]
#     rows = (n_colors + cols - 1) // cols

#     # Create a blank image to hold swatches
#     grid = np.zeros((rows * swatch_size, cols * swatch_size, 3), dtype=np.uint8)

#     for idx, color in enumerate(color_profile):
#         row = idx // cols
#         col = idx % cols
#         grid[
#             row * swatch_size : (row + 1) * swatch_size,
#             col * swatch_size : (col + 1) * swatch_size
#         ] = color.astype(np.uint8)

#     plt.figure(figsize=(cols / 2, rows / 2))
#     plt.imshow(grid)
#     plt.axis("off")
#     plt.title(f"{n_colors} Color Profile")
#     plt.show()

In [ ]:
# reference_images_downsampled, reference_color_profile = joint_kmeans_downsample(reference_images, n_colors=256)
# print(f"Reference color profile shape: {reference_color_profile.shape}")
# plot_color_profile(reference_color_profile)

In [ ]:
# # Downsample the images and update the reference_images list and dictionary
# for i, imgs in enumerate(reference_images):
#     # Downsample the image using grid method
#     downsampled_reference = pre.downsample_image(
#         imgs,
#         method="kmeans",
#         n_colors= 256
#     )

#     # Update the reference_images list with the downsampled images
#     reference_images[i] = downsampled_reference
#     reference_dict[reference_labels[i]] = downsampled_reference


# # Do the same with the train images
# for i, imgs in enumerate(train_images):
#     # Downsample the image using grid method
#     downsampled_train = pre.downsample_image(
#         imgs,
#         method="kmeans",
#         n_colors= 256
#     )

#     # Update the reference_images list with the downsampled images
#     train_images[i] = downsampled_train
#     train_dict[train_labels[i]] = downsampled_train

In [ ]:
pre.plot_reference_images(reference_images, reference_labels)

## Section 1: Pre-Processing and Background Removal For Reference Images
The goal here is to take every image and separate the background from the foreground. This will make future isoltion, cropping, and classification much easier.

In [ ]:
# ### Remove the background of all the reference images ###

# def bgremove2(myimage):
#     # First Convert to Grayscale
#     myimage_grey = cv2.cvtColor(myimage, cv2.COLOR_BGR2GRAY)
 
#     ret,baseline = cv2.threshold(myimage_grey,127,255,cv2.THRESH_TRUNC)
 
#     ret,background = cv2.threshold(baseline,126,255,cv2.THRESH_BINARY)
 
#     ret,foreground = cv2.threshold(baseline,126,255,cv2.THRESH_BINARY_INV)
 
#     foreground = cv2.bitwise_and(myimage,myimage, mask=foreground)  # Update foreground with bitwise_and to extract real foreground
 
#     # Convert black and white back into 3 channel greyscale
#     background = cv2.cvtColor(background, cv2.COLOR_GRAY2BGR)
 
#     # Combine the background and foreground to obtain our final image
#     finalimage = background+foreground
#     return finalimage

# # Apply the background removal to the reference images
# reference_images_nobg = reference_images_raw.copy()
# for i, imgs in enumerate(reference_images_raw):
#     # Apply the background removal
#     img_bgremoved = bgremove2(imgs)
    
#     # Update the reference_images list with the background removed images
#     reference_images_nobg[i] = img_bgremoved


In [ ]:
# # Plot the original image and the background removed image
# k = 3
# # Compare the original image and the background removed image
# pre.compare_2_images(
#     image1=reference_images_raw[k],
#     image2=reference_images_nobg[k],
#     title="Original vs Background Removed Image")
# print(f"Original image shape: {reference_images_raw[k].shape}")
# print(f"Background removed image shape: {reference_images_nobg[k].shape}")

#### PREPROCESS THE REFERENCE IMAGES ####

In [ ]:
### Do all of the same previous operations to th reference images to find good countours ###
# Canny Edge Detection Params
t_lower = 40
t_upper = 85 
aperture_size = 3 
L2Gradient = False

#Morphology Params
kernel_size=12
min_area_th=1
min_obj_size=100
connect=1

# Start by extracting the RGB data from all the images
N = len(reference_labels)

for i in range(N):
    label = reference_labels[i]
    img = reference_images[i]
    # Extract the RGB channels
    R, G, B = lab1.extract_rgb_channels(img)
    H, S, V = lab1.extract_hsv_channels(img)
    
    # Update the dictionary to include the RGB channels
    reference_dict[label] = {
        "image": img,  # Original image
        "R": R,  # Red channel
        "G": G,  # Green channel
        "B": B,  # Blue channel
        "H": H,  # Hue channel
        "S": S,  # Saturation channel
        "V": V   # Value channel
    }

# # Add all of the raw images to the dictionary
# for i, dict in enumerate(reference_dict.values()):
#     # Extrat Raw Image
#     img = reference_images_raw[i]

#     # Add the new key-value pair to the dictionary
#     dict["raw_image"] = img

# Convert the images to grayscale and add the grayscale images to the dictionary
for dict in reference_dict.values():
    # Extract the image from the dictionary
    img = dict["image"]
    # raw_img = dict["raw_image"]
    
    # Apply the HSV thresholding
    img_grey = pre.RGB2greyscale(img)
    # img_grey_raw = pre.RGB2greyscale(raw_img)
    
    # Add the new key-value pair to the dictionary
    dict["grey"] = img_grey
    # dict["grey_raw"] = img_grey_raw


# Apply Canny edge detection to the images and add the Canny images to the dictionary
for dict in reference_dict.values():
    # Extract the image from the dictionary
    img = np.uint8(dict["grey"])
    
    # Apply the Canny edge detection
    img_canny = cv2.Canny(img, t_lower, t_upper, apertureSize=aperture_size, L2gradient=L2Gradient)
    
    # Add the new key-value pair to the dictionary
    dict["canny"] = img_canny

# Apply morphology to the Canny edge-detected image
for dict in reference_dict.values():
    # Extract the image from the dictionary
    img = dict["canny"]
    
    # Apply the morphology
    img_morph = lab1.apply_morphology(img, kernel_size=kernel_size, min_area_th=min_area_th, min_obj_size=min_obj_size, connect=connect)
    
    # Add the new key-value pair to the dictionary
    dict["canny_morph"] = img_morph

In [9]:
### Find a single bounding box surrouning all contours in each reference image with the contours aroud the canny ###

def find_bounding_box(image: np.ndarray, contours: list) -> tuple[int, int, int, int]:
    """
    Find a single bounding box surrounding all contours in the image.

    
    Args:
        image: Input image (H, W, 3)
        contours: List of contours found in the image

    Returns:
        x_min, y_min, x_max, y_max: Coordinates of the bounding box
    """
    # Initialize min and max coordinates
    x_min = image.shape[1]
    y_min = image.shape[0]
    x_max = 0
    y_max = 0

    for contour in contours:
        # Get the bounding box for each contour
        x, y, w, h = cv2.boundingRect(contour)
        
        # Update min and max coordinates
        x_min = min(x_min, x)
        y_min = min(y_min, y)
        x_max = max(x_max, x + w)
        y_max = max(y_max, y + h)
    
    return x_min, y_min, x_max, y_max

# First pass: Find individual bounding boxes and determine max dimensions
bounding_boxes = []
max_width = 0
max_height = 0

for k in range(13):
    dict_entry = reference_dict[reference_labels[k]]
    img = dict_entry["canny"]

    # Find contours
    contours, _ = cv2.findContours(img, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    # Find bounding box around all contours
    x_min, y_min, x_max, y_max = find_bounding_box(img, contours)

    # Store bounding box
    bounding_boxes.append((x_min, y_min, x_max, y_max))

    # Update max width and height
    padding = 3
    width = x_max - x_min
    height = y_max - y_min
    max_width = max(max_width, width) + padding
    max_height = max(max_height, height) + padding

cropped_refrence_images = []
# Second pass: Re-extract bounding boxes using uniform size
for k in range(13):
    dict_entry = reference_dict[reference_labels[k]]
    og_img = reference_images[k]
    img = dict_entry["canny"]
    x_min, y_min, x_max, y_max = bounding_boxes[k]

    # Centered bounding box with uniform size
    center_x = (x_min + x_max) // 2
    center_y = (y_min + y_max) // 2

    new_x_min = max(center_x - max_width // 2, 0)
    new_y_min = max(center_y - max_height // 2, 0)
    new_x_max = min(new_x_min + max_width, og_img.shape[1])
    new_y_max = min(new_y_min + max_height, og_img.shape[0])

    # Save bounding box info
    dict_entry["contour_bounding_box"] = (new_x_min, new_y_min, new_x_max, new_y_max)

    # Extract and save cropped image
    bbox_image = og_img[new_y_min:new_y_max, new_x_min:new_x_max]
    dict_entry["bbox_image"] = bbox_image

    # Optional: draw rectangle on image
    img_with_box = img.copy()
    cv2.rectangle(img_with_box, (new_x_min, new_y_min), (new_x_max, new_y_max), (255, 0, 0), 2)
    dict_entry["image_with_box"] = img_with_box
    cropped_refrence_images.append(bbox_image)

    # print(f"Uniform Bounding Box Shape: {bbox_image.shape}")
    # Plot the bbox image
    # plt.imshow(bbox_image)
    # plt.axis("off")
    # plt.title("Bounding Box Image")
    # plt.show()



In [ ]:
### Make New Dictonary with the cropped images and Apply Preprocessing ###

# Create a new dictionary for cropped images
cropped_reference_dict = {}
for i, label in enumerate(reference_labels):
    cropped_reference_dict[label] = {
        "image": cropped_refrence_images[i],
    }

# Define Parameters
t_lower = [30, 30, 15, 30, 30, 30, 10, 30, 30, 30, 30, 10, 30]
t_upper = [115, 115, 115, 115, 115, 115, 80, 115, 115, 115, 115, 90, 115]
aperture_size = 3
L2Gradient = False

# Make the parameters dynamic to tailor the shapes
kernel_size = [6, 10, 16, 10, 12, 14, 12, 16, 4, 8, 6, 5, 8]
min_area_th = 10
min_obj_size = 20
connect = 2

# Dynamic Gaussian Blur 
blur = [5, 3, 3, 3, 9, 7, 1, 7, 5, 3, 5, 1, 5]


# Apply the same preprocessing steps to the cropped images
for i, imgs in enumerate(cropped_refrence_images):
    # Extract the RGB channels
    R, G, B = lab1.extract_rgb_channels(imgs)
    H, S, V = lab1.extract_hsv_channels(imgs)
    
    # Update the dictionary to include the RGB channels
    cropped_reference_dict[reference_labels[i]].update({
        "R": R,  # Red channel
        "G": G,  # Green channel
        "B": B,  # Blue channel
        "H": H,  # Hue channel
        "S": S,  # Saturation channel
        "V": V   # Value channel
    })

    # Convert to grayscale
    img_grey = np.uint8(pre.RGB2greyscale(imgs))
    cropped_reference_dict[reference_labels[i]]["grey"] = img_grey

    # Add a bit of gaussian blur to the images
    img_blury = cv2.GaussianBlur(img_grey, (blur[i], blur[i]), 0)
    cropped_reference_dict[reference_labels[i]]["grey_blur"] = img_blury

    # Apply Canny edge detection
    img_canny = cv2.Canny(img_blury, t_lower[i], t_upper[i], apertureSize=aperture_size, L2gradient=L2Gradient)
    cropped_reference_dict[reference_labels[i]]["canny"] = img_canny

    # Apply morphology
    img_morph = lab1.apply_morphology(img_canny, kernel_size=kernel_size[i], min_area_th=min_area_th, min_obj_size=min_obj_size, connect=connect)
    cropped_reference_dict[reference_labels[i]]["canny_morph"] = img_morph

pre.plot_ref_images_vs_filtered(cropped_reference_dict, reference_labels)




In [11]:
image_list = []
for dict in cropped_reference_dict.values():
    img = dict["canny_morph"]
    image_list.append(img)
image_array = np.stack(image_list)  # Shape: (N, H, W)

# Use your batch find_contour function
contours_list = lab2.find_contour(image_array)  # List of N arrays

In [12]:
def compute_shape_features(contour):
    area = cv2.contourArea(contour)
    perimeter = cv2.arcLength(contour, True)
    rect = cv2.boundingRect(contour)
    bounding_area = rect[2] * rect[3]
    hull = cv2.convexHull(contour)
    hull_area = cv2.contourArea(hull)
    
    compactness = (perimeter ** 2) / (4 * np.pi * area + 1e-6)
    rectangularity = area / (bounding_area + 1e-6)
    extent = area / (hull_area + 1e-6)
    
    return [area, perimeter, compactness, rectangularity, extent]

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

# Build feature matrix for reference images
reference_features = []
for contour in contours_list:
    # If multiple contours, you may want to select the largest one
    if contour is not None and contour.size > 0:
        if contour.ndim == 3:
            contour = contour.reshape(-1, 2)
        features = compute_shape_features(contour)
    else:
        features = [0, 0, 0, 0, 0]  # Or np.nan, depending on your needs
    reference_features.append(features)

reference_features = np.array(reference_features)
reference_labels_array = np.array(reference_labels)

# knn = KNeighborsClassifier(n_neighbors=3)
# knn.fit(reference_features, reference_labels_array)

# # Suppose you have a new contour from a test image
# test_contour = contours_list[0]  # Replace with your test contour
# if test_contour is not None and test_contour.size > 0:
#     if test_contour.ndim == 3:
#         test_contour = test_contour.reshape(-1, 2)
#     test_features = np.array(compute_shape_features(test_contour)).reshape(1, -1)
#     predicted_label = knn.predict(test_features)
#     print("Predicted chocolate:", predicted_label[0])
# else:
#     print("No contour found in test image.")

In [ ]:
import seaborn as sns
import pandas as pd

# Create a DataFrame for easier plotting
feature_names = ["Area", "Perimeter", "Compactness", "Rectangularity", "Extent"]
df = pd.DataFrame(reference_features, columns=feature_names)
df['Label'] = reference_labels_array

# Create a unique color palette based on number of unique labels
unique_labels = df['Label'].unique()
palette = sns.color_palette("hls", len(unique_labels))  # or try "husl", "Set3", etc.

# 2D scatter plot: Area vs Compactness
plt.figure(figsize=(8, 6))
sns.scatterplot(data=df, x="Area", y="Compactness", hue="Label", palette=palette, s=80)
plt.title("Chocolate Shape Features: Area vs Compactness")
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

# Pairplot for all features (optional)
sns.pairplot(df, hue="Label", corner=True, diag_kind="kde", palette=palette)
plt.suptitle("Pairplot of Shape Features", y=1.02)
plt.show()

In [14]:

def plot_color_list_histo(
    colors: np.ndarray,
    func: Callable,
    labels: list[str],
):
    """
    Plot scatterplots showing the distribution of color channels for a list of RGB colors.

    Args
    ----
    colors: np.ndarray (N, 3)
        Array of RGB colors, shape N x 3, values expected in 0–255 range.
    func: Callable
        Function that transforms the color list into D channels (e.g., RGB, LAB, HSV).
    labels: list of str
        Names of the channels extracted by func, e.g., ["R", "G", "B"].
    """
    if colors.ndim != 2 or colors.shape[1] != 3:
        raise ValueError("Input 'colors' must be of shape (N, 3)")

    colors = colors.astype(np.uint8)
    transformed = func(img=colors)
    C = len(transformed)

    if C != len(labels):
        raise ValueError("Number of channels returned by func must match number of labels.")

    fig, axs = plt.subplots(1, 3, figsize=(18, 5))

    axs[0].scatter(transformed[0], transformed[1], c=colors / 255.0, s=10, alpha=0.5)
    axs[0].set_xlabel(labels[0])
    axs[0].set_ylabel(labels[1])
    axs[0].set_title(f"{labels[0]} vs {labels[1]}")

    axs[1].scatter(transformed[0], transformed[2], c=colors / 255.0, s=10, alpha=0.5)
    axs[1].set_xlabel(labels[0])
    axs[1].set_ylabel(labels[2])
    axs[1].set_title(f"{labels[0]} vs {labels[2]}")

    axs[2].scatter(transformed[1], transformed[2], c=colors / 255.0, s=10, alpha=0.5)
    axs[2].set_xlabel(labels[1])
    axs[2].set_ylabel(labels[2])
    axs[2].set_title(f"{labels[1]} vs {labels[2]}")

    plt.tight_layout()
    plt.show()

def extract_rgb(img):
    return [img[:, 0], img[:, 1], img[:, 2]]

canny_morph_images = [cropped_reference_dict[label]["canny_morph"] for label in cropped_reference_dict]
original_images = [cropped_reference_dict[label]["image"] for label in cropped_reference_dict]
labels = list(cropped_reference_dict.keys())

for i, imgs in enumerate(cropped_refrence_images):

    label = reference_labels[i]
    canny_morph_image = cropped_reference_dict[reference_labels[i]]["canny_morph"]
    #pre.plot_reference_image({label: canny_morph_image}, label)

    #color_isolated_image = np.zeros((canny_morph_image[label].shape[0], canny_morph_image[label].shape[1], 3), dtype=np.uint8)
    color_isolated_image = np.zeros((canny_morph_image.shape[0], canny_morph_image.shape[1], 3), dtype=np.uint8)


    height, width = canny_morph_image.shape
    color_list = []

    for y in range(height):
        for x in range(width):
            if canny_morph_image[y, x]:
                color_isolated_image[y, x] = cropped_reference_dict[label]["image"][y, x]  # copy RGB pixel
                color_list.append(cropped_reference_dict[label]["image"][y, x])

    #pre.plot_reference_image({label: color_isolated_image}, label)

    color_swatch = np.zeros((25, 25, 3), dtype=np.uint8)

    if color_list:
        avg_color = np.mean(color_list, axis=0).astype(np.uint8)
        color_swatch = np.ones((25, 25, 3), dtype=np.uint8) * avg_color.reshape(1, 1, 3)
    
        plot_color_list_histo(colors=np.array(color_list), func=extract_rgb, labels=["R", "G", "B"])
    #pre.plot_reference_image({label: color_swatch}, label)

    #mg_blury = cv2.GaussianBlur(img_grey, (blur[i], blur[i]), 0)

    cropped_reference_dict[reference_labels[i]]["color_isolated_image"] = color_isolated_image
    cropped_reference_dict[reference_labels[i]]["color_swatch"] = color_swatch

def plot_ref_images_vs_color_isolated(reference_dict, reference_labels):
    """
    Plot the original and filtered images for each reference image.
    Args:
        reference_dict (dict): Dictionary containing reference images and their properties.
        reference_labels (list): List of labels corresponding to the reference images.
    """

    # Calculate the number of rows needed for 3 columns
    num_rows = math.ceil(len(reference_labels))

    # Create a figure with subplots
    fig, axes = plt.subplots(num_rows, 3, figsize=(18, 4.5 * num_rows))
    fig.suptitle("Filtered vs Original Images", fontsize=16)

    # Flatten the axes array for easier indexing
    axes = axes.flatten()

    for i, label in enumerate(reference_labels):
        # Original image (first column)
        original_image = reference_dict[label]['image']
        axes[i * 3].imshow(original_image)
        axes[i * 3].set_title(f"Original: {label}")
        axes[i * 3].axis('off')

        # Original image (second column)
        original_image = reference_dict[label]['color_isolated_image']
        axes[i * 3 + 1].imshow(original_image)
        axes[i * 3 + 1].set_title(f"Original: {label}")
        axes[i * 3 + 1].axis('off')

        # Color Swatch (third column)
        original_image = reference_dict[label]['color_swatch']
        axes[i * 3 + 2].imshow(original_image)
        axes[i * 3 + 2].set_title(f"Original: {label}")
        axes[i * 3 + 2].axis('off')

    # Hide any unused subplots
    for j in range(len(reference_labels) * 3, len(axes)):
        axes[j].axis('off')

    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.show()

plot_ref_images_vs_color_isolated(cropped_reference_dict, reference_labels)


In [ ]:
# Start by extracting the RGB data from all the images
N = len(train_labels)

for i in range(N):
    label = train_labels[i]
    img = train_dict[label]
    # Extract the RGB channels
    R, G, B = lab1.extract_rgb_channels(img)
    H, S, V = lab1.extract_hsv_channels(img)
    
    # Update the dictionary to include the RGB channels
    train_dict[label] = {
        "image": img,  # Original image
        "R": R,  # Red channel
        "G": G,  # Green channel
        "B": B,   # Blue channel
        "H": H,  # Hue channel
        "S": S,  # Saturation channel
        "V": V   # Value channel
    }

In [ ]:
label = train_labels[0]
lab1utils.plot_colors_histo(
    img = train_dict[label]['image'],
    func = lab1.extract_rgb_channels,
    labels = ["Red", "Green", "Blue"],
)

In [ ]:
label = train_labels[0]
lab1utils.plot_colors_histo(
    img = train_dict[label]['image'],
    func = lab1.extract_hsv_channels,
    labels = ["Hue", "Saturation", "Value"],
)

In [ ]:
### HSV Thresholding ###

# Added an "HSV" key to the dictionary to story the HSV thresholded images

for dict in train_dict.values():
    # Extract the image from the dictionary
    img = dict["image"]
    
    # Apply the HSV thresholding
    img_hsv = lab1.apply_hsv_threshold(img, H_min=0, H_max=0.2, S_min=0, S_max=1, V_min=0, V_max=0.65)
    
    # Add the new key-value pair to the dictionary
    dict["hsv_image"] = img_hsv

# Plot the original image and the thresholded image
label = train_labels[0]

# Compare the original image and the thresholded image
pre.compare_2_images(
    image1=train_dict[label]['image'],
    image2=train_dict[label]['hsv_image'],
    title="Original vs Thresholded Image")
# Compare the original image and the thresholded image




In [ ]:
# Convert the images to grayscale and add the grayscale images to the dictionary
for dict in train_dict.values():
    # Extract the image from the dictionary
    img = dict["image"]
    
    # Apply the HSV thresholding
    img_grey = np.uint8(pre.RGB2greyscale(img))
    # Add the new key-value pair to the dictionary
    dict["grey"] = img_grey

# Plot the original image and the thresholded image
label = train_labels[0]

# Compare the original image and the thresholded image
pre.compare_2_images(
    image1=train_dict[label]['image'],
    image2=train_dict[label]['grey'],
    title="Original vs Grey Image")
# Compare the original image and the thresholded image



In [ ]:
# def _color_profile_filter(
#     img: np.ndarray,
#     color_profile: np.ndarray,
#     tolerance: float = 100.0,
#     return_mask: bool = False,
#     lab_space: bool = True
# ) -> np.ndarray:
#     """
#     Filter image using chocolate color profile.
    
#     Args:
#         img: Input image (H,W,3)
#         color_profile: Array of reference colors (n_colors, 3)
#         tolerance: Maximum color distance threshold
#         return_mask: If True, returns binary mask instead of filtered image
#         lab_space: Whether to use LAB color space (recommended)
    
#     Returns:
#         Filtered image or binary mask
#     """
#     if lab_space:
#         img_space = cv2.cvtColor(img, cv2.COLOR_RGB2LAB)
#         profile_space = color_profile
#     else:
#         img_space = img
#         profile_space = color_profile
    
#     # Calculate distances to color profile
#     pixels = img_space.reshape(-1, 3)
#     distances = np.linalg.norm(pixels[:, None] - profile_space, axis=2)
#     min_dist = np.min(distances, axis=1)
#     print(f"Min distance: {min_dist}")
#     # Create mask
#     mask = (min_dist <= tolerance).reshape(img.shape[:2]).astype(np.uint8)
    
#     # Clean up mask
#     # kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5,5))
#     # mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=2)
    
#     if return_mask:
#         return mask * 255
#     else:
#         result = img.copy()
#         result[mask == 0] = [255, 255, 255]  # Set background to white
#         return result


In [ ]:
# ### Apply Color Profile Filter to Train Images and Add to Dictionary ###

# # Apply the color profile filter to the train images
# for dict in train_dict.values():
#     # Extract the image from the dictionary
#     img = dict["image"]
    
#     # Apply the color profile filter
#     img_filtered = _color_profile_filter(
#         img,
#         color_profile=reference_color_profile,
#         tolerance=40.0,
#         return_mask=False,
#         lab_space=True
#     )
    
#     # Add the new key-value pair to the dictionary
#     dict["color_filtered"] = img_filtered



In [ ]:
# Plot the original image and the filtered image
label = train_labels[0]
# Compare the original image and the filtered image
pre.compare_2_images(
    image1=train_dict[label]['image'],
    image2=train_dict[label]['color_filtered'],
    title="Original vs Filtered Image")

In [66]:
### EXECUTE PIPELINE ON TRAIN IMAGES ###
t_lower_white = 1
t_upper_white = 70
t_lower_dark = 30
t_upper_dark = 115
aperture_size = 3
L2Gradient = False

# Make the parameters dynamic to tailor the shapes
kernel_size_white = 16
kernel_size_dark = 12
min_area_th = 10
min_obj_size = 20
connect = 2

# Dynamic Gaussian Blur 
blur_white = 1
blur_dark = 5

# Number of images to process
N = 10


# Apply the same preprocessing steps to the cropped images
for i, imgs in enumerate(train_images[:N]):
    # Extract the RGB channels
    R, G, B = lab1.extract_rgb_channels(imgs)
    H, S, V = lab1.extract_hsv_channels(imgs)
    
    # Update the dictionary to include the RGB channels
    train_dict[train_labels[i]].update({
        "R": R,  # Red channel
        "G": G,  # Green channel
        "B": B,  # Blue channel
        "H": H,  # Hue channel
        "S": S,  # Saturation channel
        "V": V   # Value channel
    })

    # Convert to grayscale
    img_grey = np.uint8(pre.RGB2greyscale(imgs))
    train_dict[train_labels[i]]["grey"] = img_grey

    # Add a bit of gaussian blur to the images
    img_blury_white = cv2.GaussianBlur(img_grey, (blur_white, blur_white), 0)
    img_blury_dark = cv2.GaussianBlur(img_grey, (blur_dark, blur_dark), 0)
    train_dict[train_labels[i]]["grey_blur_white"] = img_blury_white
    train_dict[train_labels[i]]["grey_blur_dark"] = img_blury_dark

    # Apply Canny edge detection
    img_canny_white = cv2.Canny(img_blury_white, t_lower_white, t_upper_white, apertureSize=aperture_size, L2gradient=L2Gradient)
    img_canny_dark = cv2.Canny(img_blury_dark, t_lower_dark, t_upper_dark, apertureSize=aperture_size, L2gradient=L2Gradient)
    train_dict[train_labels[i]]["canny_white"] = img_canny_white
    train_dict[train_labels[i]]["canny_dark"] = img_canny_dark

    # Apply morphology
    img_morph_white = lab1.apply_morphology(img_canny_white, kernel_size=kernel_size_white, min_area_th=min_area_th, min_obj_size=min_obj_size, connect=connect)
    img_morph_dark = lab1.apply_morphology(img_canny_dark, kernel_size=kernel_size_dark, min_area_th=min_area_th, min_obj_size=min_obj_size, connect=connect)
    # for i in range(10):
    #     img_morph_dark = cv2.erode(img_morph_dark, np.ones((3, 3), np.uint8), iterations=1)
    #     img_morph_white = cv2.dilate(img_morph_white, np.ones((3, 3), np.uint8), iterations=1)

    # Apply the watershed algoithm to the dark images only

    train_dict[train_labels[i]]["canny_morph_white"] = img_morph_white
    train_dict[train_labels[i]]["canny_morph_dark"] = img_morph_dark


In [62]:
def plot_train_images_vs_filtered(train_dict, train_labels, pipeline_labels):
    """
    Plot processing pipeline steps for train images with consistent layout.
    
    Args:
        train_dict (dict): Dictionary containing train images and processed versions
        train_labels (list): List of train image labels to display
        pipeline_labels (list): Ordered list of processing steps to show as columns
                               (first should be 'image' for original)
    """
    ncols = len(pipeline_labels)
    nrows = len(train_labels)
    
    # Create figure - width scales with columns, height with rows
    fig, axes = plt.subplots(nrows, ncols, 
                           figsize=(2.5 * ncols, 2.5 * nrows))
    fig.suptitle("Image Processing Pipeline Steps", y=1.02, fontsize=14)
    
    # Handle case when there's only one row
    if nrows == 1:
        axes = axes.reshape(1, -1)
    
    for row_idx, train_label in enumerate(train_labels):
        for col_idx, step_label in enumerate(pipeline_labels):
            ax = axes[row_idx, col_idx]
            
            # Special handling for original image (color)
            if step_label == 'image':
                img = train_dict[train_label].get('image')
                if img is not None:
                    ax.imshow(img)
                ax.set_title(f"Original\n{train_label}", fontsize=10)
            
            # Handling for processed images (grayscale)
            else:
                img = train_dict[train_label].get(step_label)
                if img is not None:
                    ax.imshow(img, cmap='gray')
                ax.set_title(f"{step_label}", fontsize=10)
            
            ax.axis('off')
    
    plt.tight_layout()
    plt.show()

In [ ]:

# Define your processing steps
processing_steps = [
    'image',          # Column 1
    'canny_white',    # Column 2
    'canny_morph_white', # Column 3
    'canny_dark',        # Column 4
    'canny_morph_dark'   # Column 5
]

# Call the function
plot_train_images_vs_filtered(
    train_dict=train_dict,
    train_labels=train_labels[:N],
    pipeline_labels=processing_steps
)

In [ ]:
### Build a KNN Classifier for only the first Train Image ###
# Extract features from the first train image
train_image = train_dict[train_labels[0]]["image"]
train_image_grey = train_dict[train_labels[0]]["grey"]
train_image_canny_white = train_dict[train_labels[0]]["canny_white"]
train_image_canny_morph_white = train_dict[train_labels[0]]["canny_morph_white"]
train_image_canny_dark = train_dict[train_labels[0]]["canny_dark"]
train_image_canny_morph_dark = train_dict[train_labels[0]]["canny_morph_dark"]
# Extract contours from the white and dark images



In [ ]:
# Load image
label = train_labels[4]
image = np.uint8(train_dict[label]['grey'])

t_lower = 40
t_upper = 85 
aperture_size = 3 
L2Gradient = False

cv_canny = cv2.Canny(image, t_lower, t_upper, apertureSize=aperture_size, L2gradient=L2Gradient)

# Display results
# Display results using matplotlib
plt.figure(figsize=(10, 5))  # Set the figure size

# Plot the original image
plt.subplot(1, 2, 1)  # 1 row, 2 columns, position 1
plt.imshow(image, cmap='gray')  # Display the image in grayscale
plt.title('Original Image')
plt.axis('off')  # Turn off axis

# Plot the Canny edge-detected image
plt.subplot(1, 2, 2)  # 1 row, 2 columns, position 2
plt.imshow(cv_canny, cmap='gray')  # Display the Canny result in grayscale
plt.title('Canny Edge Detection')
plt.axis('off')  # Turn off axis

plt.tight_layout()  # Adjust spacing between subplots
plt.show()

In [ ]:
# Plot the original image and the image with the bounding box
label = reference_labels[2]
# Compare the original image and the image with the bounding box
pre.compare_2_images(
    image1=reference_dict[label]['image'],
    image2=reference_dict[label]['image_with_box'],
    title="Original vs Image with Bounding Box")

In [ ]:
# # Plot the original image and the background removed image
# k = 8
# # Compare the original image and the background removed image
# pre.compare_2_images(
#     image1=reference_images_raw[k],
#     image2=reference_images_nobg[k],
#     title="Original vs Background Removed Image")

In [ ]:
# Canny Edge Detection Params
t_lower = 40
t_upper = 85 
aperture_size = 3 
L2Gradient = False

#Morphology Params
kernel_size=12
min_area_th=1
min_obj_size=100
connect=1

In [ ]:
# Apply Canny edge detection to the images and add the Canny images to the dictionary
for dict in train_dict.values():
    # Extract the image from the dictionary
    img = np.uint8(dict["grey"])
    
    # Apply the Canny edge detection
    img_canny = cv2.Canny(img, t_lower, t_upper, apertureSize=aperture_size, L2gradient=L2Gradient)
    
    # Add the new key-value pair to the dictionary
    dict["canny"] = img_canny

# Apply morphology to the Canny edge-detected image
for dict in train_dict.values():
    # Extract the image from the dictionary
    img = dict["canny"]
    
    # Apply the morphology
    img_morph = lab1.apply_morphology(img, kernel_size=kernel_size, min_area_th=min_area_th, min_obj_size=min_obj_size, connect=connect)
    
    # Add the new key-value pair to the dictionary
    dict["canny_morph"] = img_morph

# Plot the original image and the thresholded image
label = train_labels[6]
# Compare the original image and the thresholded image
pre.compare_2_images(
    image1=train_dict[label]['canny'],
    image2=train_dict[label]['canny_morph'],
    title="Canny vs Morphology Image")


In [ ]:
label = train_labels[20]
pre.compare_2_images(
    image1=train_dict[label]['canny'],
    image2=train_dict[label]['canny_morph'],
    title="Canny vs Morphology Image")

In [ ]:
k = 2
img = reference_images_raw[k]
grey = pre.RGB2greyscale(img)
canny = cv2.Canny(img, t_lower, t_upper, apertureSize=aperture_size, L2gradient=L2Gradient)
morph = lab1.apply_morphology(canny, kernel_size=kernel_size, min_area_th=min_area_th, min_obj_size=min_obj_size, connect=connect)
# Compare the original image and the thresholded image
pre.compare_2_images(
    image1=img,
    image2=grey,
    title="Original vs Grey Image")
pre.compare_2_images(
    image1=canny,
    image2=morph,
    title="Canny vs. Morph")



In [ ]:
def find_contour(image: np.ndarray):
    """
    Find the contours for a single image.

    Args
    ----
    image: np.ndarray (H, W)
        Source image to process (binary image).

    Return
    ------
    contours: list of np.ndarray
        List of arrays containing the coordinates of the contours. Each element of the 
        list is an array of 2D coordinates (K, 2) where K depends on the number of elements 
        that form the contour.
    """
    # Ensure the input image is binary
    binary_image = (image > 0).astype(np.uint8)

    # Find contours using OpenCV
    contours, _ = cv2.findContours(binary_image, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    # Convert contours to a list of arrays with shape (K, 2)
    contours = [contour.squeeze() for contour in contours if contour.size > 0]

    return contours

def simple_segmentation(image):
    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)

    # Thresholds for neutral/light backgrounds — adjust as needed
    lower = np.array([0, 0, 0])
    upper = np.array([100, 10, 180])

    # Create binary mask: 0 for background, 255 for foreground
    background_mask = cv2.inRange(hsv, lower, upper)
    foreground_mask = cv2.bitwise_not(background_mask)

    # Optional: morphological cleanup
    kernel = np.ones((5, 5), np.uint8)
    cleaned = cv2.morphologyEx(foreground_mask, cv2.MORPH_OPEN, kernel)

    return cleaned

In [ ]:
### Find Contours of all Reference Images and add them to the Dictionary ###

for dict in reference_dict.values():
    # Extract the image from the dictionary
    img = dict["canny_morph"]
    
    # Find contours
    contours = find_contour(img)
    
    # Add the new key-value pair to the dictionary
    dict["contours"] = contours

for dict in train_dict.values():
    # Extract the image from the dictionary
    img = dict["canny_morph"]
    
    # Find contours
    contours = find_contour(img)
    
    # Add the new key-value pair to the dictionary
    dict["contours"] = contours





In [ ]:
# Plot the image with the contour on it
label = train_labels[50]
img = train_dict[label]['image']
contours = train_dict[label]['contours']

# Create a copy of the image to draw contours on
img_with_contours = img.copy()
# Draw contours on the image
for contour in contours:
    cv2.drawContours(img_with_contours, [contour], -1, (0, 255, 0), 2)  # Green color and thickness of 2
# Display the image with contours
plt.figure(figsize=(10, 5))
plt.imshow(img_with_contours)
plt.title('Image with Contours')
plt.axis('off')  # Turn off axis
plt.show()

In [ ]:
# # Get the comtesse Contour #
# img = reference_dict['Comtesse']['canny']
# contour = find_contour(img)
# print(f"Number of contours found: {len(contour)}")
# print(f"Contour shape: {contour[0].shape}")

# # Draw the contours on the original image
# img_with_contours = reference_dict['Comtesse']['image'].copy()
# # Draw only the first contour on the image

# cv2.drawContours(img_with_contours, [contour[2]], -1, (0, 255, 0), 2)  # Green color and thickness of 2
# cv2.drawContours(img_with_contours, [contour[3]], -1, (0, 255, 0), 2)  # Green color and thickness of 2

# # Display the image with contours
# plt.figure(figsize=(10, 5))
# plt.imshow(img_with_contours)
# plt.title('Image with Contours')
# plt.axis('off')  # Turn off axis
# plt.show()

# # Add the contours to the dictionary
# reference_dict['Comtesse']['contours'] = contour

In [ ]:
### Find the Bounding Boxes of the Contours for the Reference Images and Add them to the Dictionary ###

def find_bounding_boxes(image: np.ndarray, contours: list[np.ndarray]) -> list[np.ndarray]:
    """ Takes an image and its contour and returns the image within the bounding box of the contour.
    Args:
        contours: List of contours (each contour is a list of points)
    Returns:
        bbox_image: Image within the bounding box of the contour
    """
    # Save the bounding box images and bounding boxes
    bbox_images = []
    bboxes = []

    for contour in contours:
        # Compute the bounding box for the contour
        x, y, w, h = cv2.boundingRect(contour)
  
        # Extract the section of the image within the bounding box
        bbox_image = image[y:y+h, x:x+w]
        
        # Add the extracted section to the list
        bbox_images.append(bbox_image)
        bboxes.append((x, y, w, h))
    
    return bbox_images, bboxes


In [ ]:
### Add the raw image to the dictionary ###
for i, dict in enumerate(reference_dict.values()):
    # Extract the image from the dictionary
    raw_img = reference_images_raw[i]
    
    # Add the new key-value pair to the dictionary
    dict["raw_image"] = img

In [ ]:
### Find the bounding boxes for the reference images and add them to the dictionary ###
for dict in reference_dict.values():
    # Extract the image and contours from the dictionary
    img = dict["image"]
    contours = dict["contours"]
    
    # Find bounding boxes
    bbox_images, bboxes = find_bounding_boxes(img, contours)
    
    # Add the new key-value pair to the dictionary
    dict["bbox_images"] = bbox_images
    dict["bboxes"] = bboxes

# Collect bbox_images for all labels in reference_labels
bbox_images = [reference_dict[label]['bbox_images'] for label in reference_labels]


In [ ]:
# Plot the first bbox image
plt.figure(figsize=(10, 5))
plt.imshow(bbox_images[1][0])
plt.title('Bounding Box Image')
plt.axis('off')  # Turn off axis
plt.show()

In [ ]:
### Extract Color Profile from the Bounding Box Images ###
# Flatten and concatenate all pixels from all images
pixel_arrays = [img.reshape(-1, 3) for bbox in bbox_images for img in bbox]

cropped_images, cropped_color_profile = joint_kmeans_downsample(pixel_arrays, n_colors=256)
print(f"Reference color profile shape: {cropped_color_profile.shape}")
plot_color_profile(cropped_color_profile)

In [ ]:
def get_color_profile(reference_images):
    """
    Compute the unique RGB colors and their frequencies across all reference images using NumPy.

    Args:
        reference_images (List[np.ndarray]): List of (H, W, 3) RGB images

    Returns:
        unique_colors (np.ndarray): (N, 3) array of unique colors
        frequencies (np.ndarray): (N,) array of color frequencies
    """
    # Stack all image pixels into one big (total_pixels, 3) array
    all_pixels = np.concatenate([img.reshape(-1, 3) for img in reference_images], axis=0)

    # Convert to tuple-like for uniqueness check
    pixels_view = all_pixels.view([('', all_pixels.dtype)] * 3)

    # Get unique colors and their counts
    unique_pixels, counts = np.unique(pixels_view, return_counts=True)

    # Convert back to (N, 3) array
    unique_colors = unique_pixels.view(all_pixels.dtype).reshape(-1, 3)

    return unique_colors, counts

In [ ]:
def get_colors_inside_contours(reference_images: list[np.ndarray], contours: list[np.ndarray]) -> tuple[np.ndarray, np.ndarray]:
    """
    Compute the unique RGB colors and their frequencies for the pixels inside the contours of the reference images.

    Args:
        reference_images (list[np.ndarray]): List of (H, W, 3) RGB images.
        contours (list[np.ndarray]): List of contours (one contour per image).

    Returns:
        unique_colors (np.ndarray): (N, 3) array of unique colors.
        frequencies (np.ndarray): (N,) array of color frequencies.
    """
    all_pixels_inside_contours = []

    for image, contour in zip(reference_images, contours):
        # Create a mask for the current contour
        mask = np.zeros(image.shape[:2], dtype=np.uint8)  # Create a blank mask (same height and width as the image)
        cv2.drawContours(mask, [contour], -1, 255, thickness=cv2.FILLED)  # Fill the contour on the mask

        # Extract the pixels inside the contour using the mask
        pixels_inside_contour = image[mask == 255]  # Get only the pixels where the mask is 255
        all_pixels_inside_contours.append(pixels_inside_contour)

    # Stack all pixels from all images into one array
    all_pixels_inside_contours = np.vstack(all_pixels_inside_contours)

    # Get unique colors and their counts
    pixels_view = all_pixels_inside_contours.view([('', all_pixels_inside_contours.dtype)] * 3)  # Convert to tuple-like for uniqueness
    unique_pixels, counts = np.unique(pixels_view, return_counts=True)  # Get unique colors and their counts

    # Convert back to (N, 3) array
    unique_colors = unique_pixels.view(all_pixels_inside_contours.dtype).reshape(-1, 3)

    return unique_colors, counts

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches

def plot_color_histogram(unique_colors: np.ndarray, frequencies: np.ndarray, sort: bool = True, top_n: int = 100):
    """
    Plot a histogram of color frequencies using actual color bars.

    Args:
        unique_colors (np.ndarray): (N, 3) array of RGB colors
        frequencies (np.ndarray): (N,) array of corresponding frequencies
        sort (bool): Whether to sort colors by frequency
        top_n (int): Max number of colors to show (for legibility)
    """
    if sort:
        sorted_idx = np.argsort(frequencies)[::-1]
        unique_colors = unique_colors[sorted_idx]
        frequencies = frequencies[sorted_idx]

    if top_n:
        unique_colors = unique_colors[:top_n]
        frequencies = frequencies[:top_n]

    fig, ax = plt.subplots(figsize=(16, 4))
    bar_width = 1.0 / len(unique_colors)

    for i, (color, freq) in enumerate(zip(unique_colors, frequencies)):
        color_hex = tuple(c / 255 for c in color)
        rect = patches.Rectangle(
            (i * bar_width, 0), bar_width, freq,
            facecolor=color_hex, edgecolor='black'
        )
        ax.add_patch(rect)

    ax.set_xlim(0, 1)
    ax.set_ylim(0, max(frequencies) * 1.1)
    ax.axis('off')
    plt.title("Color Frequency Histogram")
    plt.tight_layout()
    plt.show()


In [ ]:
unique_colors, counts = get_color_profile(reference_images_raw)
print(f"Unique colors shape: {unique_colors.shape}")

In [ ]:
### Apply Color Profile Filter to Train Images and Add to Dictionary ###

# Apply the color profile filter to the train images
for dict in train_dict.values():
    # Extract the image from the dictionary
    img = dict["image"]
    
    # Apply the color profile filter
    img_filtered = _color_profile_filter(
        img,
        color_profile=cropped_color_profile,
        tolerance=50.0,
        return_mask=False,
        lab_space=True
    )
    
    # Add the new key-value pair to the dictionary
    dict["color_filtered"] = img_filtered


In [ ]:
# Plot the original image and the filtered image
label = train_labels[20]
# Compare the original image and the filtered image
pre.compare_2_images(
    image1=train_dict[label]['image'],
    image2=train_dict[label]['color_filtered'],
    title="Original vs Filtered Image")

In [ ]:
# Load image
image = reference_images[0]

# Create mask
mask = simple_segmentation(image)

# Create a blue background (same shape as image)
blue_background = np.full_like(image, (255, 0, 0))  # BGR for blue

# Combine chocolates with blue background
foreground = cv2.bitwise_and(image, image, mask=mask)
background = cv2.bitwise_and(blue_background, blue_background, mask=cv2.bitwise_not(mask))
result = cv2.add(foreground, background)

# Convert to RGB for display
original_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
result_rgb = cv2.cvtColor(result, cv2.COLOR_BGR2RGB)

# Display side-by-side
plt.figure(figsize=(10, 5))

plt.subplot(1, 2, 1)
plt.imshow(original_rgb)
plt.title("Original Image")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(result_rgb)
plt.title("Background Replaced with Blue")
plt.axis("off")

plt.tight_layout()
plt.show()
import cv2
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
def get_binary_mask(image):
    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)

    # Threshold for neutral/light backgrounds
    lower = np.array([0, 0, 160])
    upper = np.array([180, 40, 255])

    background_mask = cv2.inRange(hsv, lower, upper)
    foreground_mask = cv2.bitwise_not(background_mask)

    # Morphological cleanup (optional)
    kernel = np.ones((5, 5), np.uint8)
    cleaned = cv2.morphologyEx(foreground_mask, cv2.MORPH_OPEN, kernel)

    # Convert to binary: 0 = foreground (chocolates), 1 = background
    binary_image = np.where(cleaned > 0, 0, 1).astype(np.uint8)

    return binary_image


In [ ]:
binary_mask = get_binary_mask(image)

# Display binary mask
plt.imshow(binary_mask, cmap='gray')
plt.title("Binary Image (0=Object, 1=Background)")
plt.axis('off')
plt.show()
mask = simple_segmentation(np.uint8(image))
result = cv2.bitwise_and(image, image, mask=mask)
# Convert to grayscale and preprocess (e.g., thresholding)
gray_image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
_, binary_image = cv2.threshold(gray_image, 127, 255, cv2.THRESH_BINARY)

# Find contours
contours = find_contour(binary_image)

# Draw contours in red on the original image
output_image = image.copy()
cv2.drawContours(output_image, contours, -1, (0, 0, 255), 2)  # Red color (BGR: (0, 0, 255)), thickness=2

# Display the result
plt.figure(figsize=(8, 8))
plt.imshow(cv2.cvtColor(output_image, cv2.COLOR_BGR2RGB))  # Convert BGR to RGB for matplotlib
plt.title("Contours in Red")
plt.axis('off')
plt.show()


In [ ]:
# clear the variable called dict
del dict

In [ ]:
import cv2
import numpy as np
from matplotlib import pyplot as plt

class EnhancedTruffleIdentifier:
    def __init__(self, reference_images, reference_labels, downscale_factor=1.0):
        """
        Args:
            reference_images: List of preprocessed reference images
            reference_labels: List of truffle type names  
            downscale_factor: Resize factor for faster processing (0.25 = 1/4 size)
        """
        self.sift = cv2.SIFT_create(
            nfeatures=2000,  # Reduced for downscaled images
            contrastThreshold=0.02,  # Slightly higher for stability
            edgeThreshold=10,
            sigma=1.0
        )
        
        # FLANN parameters
        self.index_params = dict(algorithm=1, trees=5)
        self.search_params = dict(checks=100)
        self.matcher = cv2.FlannBasedMatcher(self.index_params, self.search_params)
        
        # Store downscaled references
        self.ref_kps = []
        self.ref_des = [] 
        self.labels = []
        self.downscale_factor = downscale_factor
        
        for img, label in zip(reference_images, reference_labels):
            # Downscale reference image
            h, w = img.shape[:2]
            new_size = (int(w*downscale_factor), int(h*downscale_factor))
            img_small = cv2.resize(img, new_size)
            
            # Convert to grayscale and mask white background
            gray = cv2.cvtColor(img_small, cv2.COLOR_RGB2GRAY)
            _, mask = cv2.threshold(gray, 250, 255, cv2.THRESH_BINARY_INV)
            
            # Detect features
            kp, des = self.sift.detectAndCompute(gray, mask)
            
            if des is not None and len(des) > 10:  # Only keep if enough features
                self.ref_kps.append(kp)
                self.ref_des.append(des)
                self.labels.append(label)
                print(f"Loaded {label} with {len(des)} features")
            else:
                print(f"WARNING: {label} has insufficient features ({len(des) if des is not None else 0})")

    def identify_truffles(self, query_img, min_matches=10, reproj_thresh=3.0, match_ratio=0.8):
        """More robust matching with adjustable parameters"""
        # Downscale query image
        h, w = query_img.shape[:2]
        new_size = (int(w*self.downscale_factor), int(h*self.downscale_factor))
        query_small = cv2.resize(query_img, new_size)
        query_gray = cv2.cvtColor(query_small, cv2.COLOR_RGB2GRAY)
        
        # Detect query features
        query_kp, query_des = self.sift.detectAndCompute(query_gray, None)
        if query_des is None:
            print("No features detected in query image")
            return []
        
        results = []
        
        for i, (ref_des, label) in enumerate(zip(self.ref_des, self.labels)):
            try:
                matches = self.matcher.knnMatch(query_des, ref_des, k=2)
                
                # Lowe's ratio test with relaxed threshold
                good = []
                for m, n in matches:
                    if m.distance < match_ratio * n.distance:
                        good.append(m)
                
                if len(good) >= min_matches:
                    src_pts = np.float32([query_kp[m.queryIdx].pt for m in good])
                    dst_pts = np.float32([self.ref_kps[i][m.trainIdx].pt for m in good])
                    
                    # Find homography with RANSAC
                    M, mask = cv2.findHomography(dst_pts, src_pts, cv2.RANSAC, reproj_thresh)
                    
                    if mask is not None:
                        inliers = np.sum(mask)
                        if inliers >= min_matches:
                            # Scale bounding box back to original size
                            h_ref, w_ref = reference_images[i].shape[:2]
                            ref_corners = np.float32([[0,0], [0,h_ref-1], [w_ref-1,h_ref-1], [w_ref-1,0]])
                            query_corners = cv2.perspectiveTransform(
                                ref_corners.reshape(-1,1,2), 
                                M * (1/self.downscale_factor)  # Scale transform
                            )
                            
                            # Create visualization (on downscaled images for speed)
                            vis = cv2.drawMatches(
                                query_small, query_kp,
                                cv2.resize(reference_images[i], new_size), self.ref_kps[i],
                                [m for m,valid in zip(good,mask) if valid],
                                None,
                                flags=cv2.DRAW_MATCHES_FLAGS_NOT_DRAW_SINGLE_POINTS
                            )
                            
                            results.append((
                                label,
                                cv2.boundingRect(query_corners),
                                vis
                            ))
            except Exception as e:
                print(f"Error matching {label}: {str(e)}")
                continue
                
        return results

# Usage Example
identifier = EnhancedTruffleIdentifier(cropped_refrence_images, reference_labels)

test_img = train_images[2]  # Use your test image
test_img_rgb = cv2.cvtColor(test_img, cv2.COLOR_BGR2RGB)

results = identifier.identify_truffles(
    test_img_rgb,
    min_matches=8,            # Reduced from 15
    reproj_thresh=4.0,        # Increased from 2.5  
    match_ratio=0.8           # More lenient than 0.7
)

# Visualization
if len(results) > 0:
    for label, bbox, vis in results:
        x,y,w,h = bbox
        output_img = test_img_rgb.copy()
        cv2.rectangle(output_img, (x,y), (x+w,y+h), (0,255,0), 10)
        cv2.putText(output_img, label, (x,y-20), 
                   cv2.FONT_HERSHEY_SIMPLEX, 2, (0,255,0), 3)
        
        plt.figure(figsize=(20,10))
        plt.subplot(121), plt.imshow(vis)
        plt.title(f"Feature Matches: {label}"), plt.axis('off')
        plt.subplot(122), plt.imshow(output_img)
        plt.title("Detection Result"), plt.axis('off')
        plt.show()
else:
    print("No matches found - try these adjustments:")
    print("1. Increase downscale_factor (up to 0.5)")
    print("2. Decrease min_matches (down to 5)") 
    print("3. Increase reproj_thresh (up to 5.0)")
    print("4. Increase match_ratio (up to 0.85)")